# FLUX.2 Klein-4B on TPU

Runner for [JohanesSetiawan/flux2-tpu](https://github.com/JohanesSetiawan/flux2-tpu).

This notebook contains no logic beyond the bootstrap cell below. Every
other cell is a call into the `src` package, where the work lives and is
tested. If you find yourself writing an algorithm in a cell here, it
belongs in the package instead.

**Before running:** select a TPU accelerator. On Kaggle that is under
Settings, Accelerator.

**No safety filtering.** This codebase includes no content moderation or
output filtering of any kind. You are responsible for what you generate.

## 1. Set up the repository

Safe to re-run, and safe to run alone after a kernel restart.

A restart resets the working directory and clears `sys.path`, so a cell
that only changed directory during the first run would leave `src`
unimportable afterwards. This cell therefore clones only if the
repository is missing, and always sets both the directory and the import
path. Running it after a restart is enough to make every later cell work
again, without re-downloading anything.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPOSITORY_URL = "https://github.com/JohanesSetiawan/flux2-tpu.git"
REPOSITORY_DIRECTORY = Path("/kaggle/working/flux2-tpu")

if not REPOSITORY_DIRECTORY.exists():
    subprocess.run(
        ["git", "clone", "--depth", "1", REPOSITORY_URL, str(REPOSITORY_DIRECTORY)],
        check=True,
    )
else:
    print(f"{REPOSITORY_DIRECTORY} already present; not cloning again")

os.chdir(REPOSITORY_DIRECTORY)

# Prepending rather than appending, so this repository wins over any
# same-named package that happens to be installed in the environment.
if str(REPOSITORY_DIRECTORY) not in sys.path:
    sys.path.insert(0, str(REPOSITORY_DIRECTORY))

import src  # noqa: F401  Fail here, loudly, rather than three cells later.

print(f"working directory: {Path.cwd()}")
print("src package is importable")

## 2. Install dependencies

Also safe to re-run. Skip it after a restart if the session has not been
recycled, since the packages are still installed.

In [ ]:
%pip install --quiet -r requirements.txt
%pip install --quiet ipywidgets gradio pillow

## 3. Configure

`compilation_cache_directory` is worth setting on a hosted notebook:
compiled programs then survive a restart, which otherwise costs a full
recompilation every time.

In [ ]:
import logging
from pathlib import Path

from src.config import (
    CheckpointSourceConfig,
    ExecutionConfig,
    InferenceConfig,
    MemoryResidencyStrategy,
)
from src.utils import configure_logging

WORKING_DIRECTORY = Path("/kaggle/working")

logger = configure_logging(
    log_file_path=WORKING_DIRECTORY / "generation_log.txt",
    logger_name="flux2_klein",
)

inference_config = InferenceConfig(
    checkpoint_source=CheckpointSourceConfig(
        local_cache_directory=Path("/kaggle/temp/flux2_klein_checkpoint_cache"),
    ),
    # AUTO picks a residency strategy from the visible device count: a
    # single chip keeps the transformer and decoder resident and swaps
    # the text encoder, a pod keeps everything.
    residency_strategy=MemoryResidencyStrategy.AUTO,
)

execution_config = ExecutionConfig(
    compilation_cache_directory=WORKING_DIRECTORY / "compilation_cache",
)

## 4. Load

Downloads roughly 11 GB the first time and restores three components.
Later runs reuse the local cache, so this is fast after the first.

In [ ]:
from src.pipeline import Pipeline

pipeline = Pipeline(inference_config, logger, execution_config=execution_config)
pipeline.load()

## 5. Warm up

Compilation is per output shape, not per prompt, so this pays the
compile cost once per resolution rather than on the first request that
uses it. Expect this to be the slowest cell on a cold cache.

In [ ]:
pipeline.warm_up()

## 6. Generate

Three ways into the same pipeline. They share all their input handling,
so they behave identically.

A seed of `-1` draws a random one. The seed actually used is reported
after each generation, so a result you like can be reproduced.

### Option A: in-notebook controls

In [ ]:
from IPython.display import display

from src.interfaces.widgets import build_control_panel

display(build_control_panel(pipeline, logger))

### Option B: browser interface

`share=True` creates a public link, which is usually how a hosted
notebook is reached. That link is public while the cell runs; leave it
off if you would rather not expose the interface.

In [ ]:
from src.interfaces.browser import build_interface

build_interface(pipeline, logger).launch(share=True)

### Option C: call it directly

In [ ]:
from PIL import Image

from src.interfaces.session import build_request, to_display_image

request = build_request(
    prompt="a lighthouse on a rocky shore at dusk",
    resolution_label="1024x1024",
    requested_seed=-1,
    buckets=pipeline.resolution_buckets,
)

image = pipeline.generate(request)
print(f"seed {request.seed}")
Image.fromarray(to_display_image(image))

## Notes

**After a kernel restart**, run cell 1 again before anything else. It
will not re-clone or re-download; it only restores the working directory
and the import path, which a restart clears.

Repeated prompts skip the text encoder: conditioning is cached by prompt
text, so changing only the seed or resolution reuses it.

Only three resolutions are offered, deliberately. Above roughly 4300
image tokens the reference sampling schedule switches to a formula
derived for a two-hundred-step model, which this four-step checkpoint
was not tuned against. See AGENTS.md for the measured discontinuity.

The full log is written to `generation_log.txt` in the working
directory and survives the notebook output being cleared.